In [6]:
import pandas as pd
import altair as alt
import numpy as np
import theme

alt.themes.register('main_theme', theme.main_theme)
alt.themes.enable('main_theme')

ThemeRegistry.enable('main_theme')

In [7]:
# read in 60y entropy of sites
entropy_df = pd.concat(
    [pd.read_csv(
        'data/nextstrain_groups_blab_flu_seasonal_h3n2_ha1_60y_diversity.tsv', sep = '\t'
    ),
    pd.read_csv(
        'data/nextstrain_groups_blab_flu_seasonal_h3n2_ha2_60y_diversity.tsv', sep = '\t'
    ).assign(position=lambda x: x['position'] + 329)]
).rename(
    columns={'position': 'site'}
).drop(columns=['gene'])

entropy_df.head()

,site,entropy
0,1,0.034
1,2,0.271
2,3,0.719
3,4,0.032
4,5,0.169


In [8]:
import json
import pandas as pd
from pathlib import Path

json_file = Path("data/alignments/nodeSeqs_nuc.fasta.FEL.json")

with json_file.open() as fh:
    fel = json.load(fh)

# Grab the raw matrix
raw = fel["MLE"]["content"]["0"]

# Build a DataFrame
cols = ["alpha", "beta", "alpha=beta", "LRT", "p-value", "branch_length"]
df   = pd.DataFrame(raw, columns=cols)

# Add convenience columns
df["site"]   = df.index - 15 # 1‑based codon index
df["dN/dS"]  = df["beta"] / df["alpha"] # ω

df = df.assign(**{"p-value": df["p-value"] + 1e-15}) # pseudocount
df.head()

,alpha,beta,alpha=beta,LRT,p-value,branch_length,site,dN/dS
0,0.000000,0.000000e+00,0.000000,0.000000,1.000000e+00,0.000000,-15,NaN
1,0.190768,2.884963e-01,0.256323,0.252194,6.155348e-01,2.435822,-14,1.512286
2,2.943484,1.993304e-01,0.990730,39.254368,3.720362e-10,9.414861,-13,0.067719
3,0.438087,2.784153e-01,0.332155,0.442682,5.058307e-01,3.156453,-12,0.635526
4,0.143986,1.974251e-07,0.038617,2.632981,1.046650e-01,0.366976,-11,0.000001


In [9]:
merged_df = pd.merge(df, entropy_df, on='site', how='left')
merged_df = merged_df.assign(
    log10P=np.select(
        [
            merged_df["dN/dS"] < 1,
            merged_df["dN/dS"] > 1
        ],
        [
            np.log10(merged_df["p-value"]),
            -np.log10(merged_df["p-value"])
        ],
        default=np.nan
    )
)
merged_df.head()

,alpha,beta,alpha=beta,LRT,p-value,branch_length,site,dN/dS,entropy,log10P
0,0.000000,0.000000e+00,0.000000,0.000000,1.000000e+00,0.000000,-15,NaN,NaN,NaN
1,0.190768,2.884963e-01,0.256323,0.252194,6.155348e-01,2.435822,-14,1.512286,NaN,0.210747
2,2.943484,1.993304e-01,0.990730,39.254368,3.720362e-10,9.414861,-13,0.067719,NaN,-9.429415
3,0.438087,2.784153e-01,0.332155,0.442682,5.058307e-01,3.156453,-12,0.635526,NaN,-0.295995
4,0.143986,1.974251e-07,0.038617,2.632981,1.046650e-01,0.366976,-11,0.000001,NaN,-0.980198


In [10]:
# altair scatter plot of dN/dS vs. entropy
scatter = alt.Chart(merged_df).mark_circle(size=50).encode(
    x=alt.X('log10P', title='log10P'),
    y=alt.Y('entropy', title='Entropy'),
    tooltip=['site', 'dN/dS', 'alpha', 'beta', 'entropy', 'log10P']
).properties(
    width=200,
    height=200,
)
scatter.display()

alt.Chart(...)

In [11]:
# read in structure mapping
site_map = pd.read_csv('../data/site_numbering_map.csv')
site_map.head()

,sequential_site,reference_site,sequential_wt,region,rbs_region
0,1,1,Q,HA1,outside RBS
1,2,2,K,HA1,outside RBS
2,3,3,I,HA1,outside RBS
3,4,4,P,HA1,outside RBS
4,5,5,G,HA1,outside RBS


In [12]:
site_info_df = pd.merge(
    merged_df, 
    site_map[['reference_site', 'region', 'rbs_region']], 
    left_on='site', 
    right_on='reference_site',
    how='left'
)
site_info_df.head()

,alpha,beta,alpha=beta,LRT,p-value,branch_length,site,dN/dS,entropy,log10P,reference_site,region,rbs_region
0,0.000000,0.000000e+00,0.000000,0.000000,1.000000e+00,0.000000,-15,NaN,NaN,NaN,NaN,NaN,NaN
1,0.190768,2.884963e-01,0.256323,0.252194,6.155348e-01,2.435822,-14,1.512286,NaN,0.210747,NaN,NaN,NaN
2,2.943484,1.993304e-01,0.990730,39.254368,3.720362e-10,9.414861,-13,0.067719,NaN,-9.429415,NaN,NaN,NaN
3,0.438087,2.784153e-01,0.332155,0.442682,5.058307e-01,3.156453,-12,0.635526,NaN,-0.295995,NaN,NaN,NaN
4,0.143986,1.974251e-07,0.038617,2.632981,1.046650e-01,0.366976,-11,0.000001,NaN,-0.980198,NaN,NaN,NaN


In [13]:
# Define custom colors for each RBS region
colors = {
    'outside RBS': '#bdbebb',
    '130-loop': '#725663',
    '150-loop': '#725663',
    '190-helix': '#725663',
    '220-loop': '#725663',
    'RBS base': '#725663'
}

median_dnds_rbs = site_info_df[['site', 'rbs_region', 'dN/dS']].drop_duplicates().groupby(
    "rbs_region"
)["dN/dS"].median().reset_index().rename(
    columns={'dN/dS': 'median dN/dS'}
).query(
    'rbs_region != "RBS other"'
)

bar_chart = alt.Chart(median_dnds_rbs).mark_bar(size=20).encode(
    x=alt.X(
        'median dN/dS:Q',
        title=['Median dN/dS', '(in natural sequences)'],
        axis=alt.Axis(
            tickCount=2,
        )
    ),
    y=alt.Y(
        'rbs_region:N',
        title=None,
    ),
    color=alt.Color(
            "rbs_region",
            scale=alt.Scale(domain=list(colors.keys()), range=list(colors.values())),
            legend=None
    ),
    tooltip=['rbs_region', 'median dN/dS']
).properties(
    height=200,
    width=80
)

bar_chart

alt.Chart(...)

In [14]:
# Define custom colors for each RBS region
colors = {
    'epitope-A': '#FFB547',
    'epitope-B': '#FFB547',
    'epitope-C': '#FFB547',
    'epitope-D': '#FFB547',
    'epitope-E': '#FFB547',
    'HA1': '#bdbebb',
    'HA2': '#767676',
}

order = ['epitope-A', 'epitope-B', 'epitope-C', 'epitope-D', 'epitope-E', 'HA1', 'HA2']

median_dnds_epitope = site_info_df[['site', 'region', 'dN/dS']].drop_duplicates().groupby(
    "region"
)["dN/dS"].median().reset_index().rename(
    columns={'dN/dS': 'median dN/dS'}
)

bar_chart = alt.Chart(median_dnds_epitope).mark_bar(size=20).encode(
    x=alt.X(
        'median dN/dS:Q',
        title=['Median dN/dS', '(in natural sequences)'],
        axis=alt.Axis(
            tickCount=1,
        )
    ),
    y=alt.Y(
        'region:N',
        sort=order, 
        title=None,
    ),
    color=alt.Color(
            "region",
            scale=alt.Scale(domain=list(colors.keys()), range=list(colors.values())),
            legend=None
    ),
    tooltip=['region', 'median dN/dS']
).properties(
    height=200,
    width=80
)

bar_chart

alt.Chart(...)